In [1]:
import torch 
import torch.nn as nn
import numpy as np

# Create Model

In [25]:
class DeepONets(nn.Module):
    def __init__(self, branch_layers, trunk_layers):
        super(DeepONets, self).__init__()
        self.branch_layers = branch_layers
        self.trunk_layers = trunk_layers

        self.branch_model = nn.Sequential()
        self.trunk_model = nn.Sequential()

    def initial_branch_model(self):
        for i in range(len(self.branch_layers) - 1):
            self.branch_model.add_module("layer{}".format(i), nn.Linear(self.branch_layers[i], self.branch_layers[i+1]))
            self.branch_model.add_module("act{}".format(i), nn.Tanh())
        self.branch_model.add_module("end", nn.Linear(self.branch_layers[-1], self.branch_layers[-1]))

    def initial_trunk_model(self):
        for i in range(len(self.trunk_layers) - 1):
            self.trunk_model.add_module("layer{}".format(i), nn.Linear(self.trunk_layers[i], self.trunk_layers[i+1]))
            self.trunk_model.add_module("act{}".format(i), nn.Tanh())
        self.trunk_model.add_module("end", nn.Linear(self.trunk_layers[-1], self.trunk_layers[-1]))

    def forward(self, y, x):
        branch_output = self.branch_model(y)
        trunk_output = self.trunk_model(x)

        output = torch.sum(branch_output * trunk_output, dim=1)

        return output

# Derivative

In [23]:
def grad(outputs, inputs):
    return torch.autograd.grad(outputs, inputs, grad_outputs=torch.ones_like(outputs), create_graph=True)

In [28]:
# Initialize model
branch_layers = [100, 50, 50, 50, 50, 50] 
trunk_layers =  [1, 50, 50, 50, 50, 50]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = DeepONets(branch_layers, trunk_layers).to(device)
model.initial_branch_model()
model.initial_trunk_model()

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# print(model.trunk_model)

# Data Generation

In [ ]:
N_train = 10000
m = 100
P_train = 1